# House Price MLOps — All in One, dari awal

Notebook ini menulis seluruh alur langsung di dalam notebook. Tidak ada import dari `house_price` atau `src/`. Gunakan notebook ini untuk belajar dari awal; `experiments.ipynb` yang satunya memperlihatkan versi modular.

Urutan praktik: **environment → data → DVC → validasi → split → preprocessing → tuning → evaluasi → MLflow → load → prediksi**. Bagian akhir menjelaskan cara menghubungkan hasilnya ke Git, Docker, dan GitHub Actions.

Jalankan cell berurutan dari atas. Mode cepat memakai seluruh dataset dengan grid kecil; grid lengkap dapat dipilih melalui `QUICK_DEMO`. Dataset California Housing adalah data historis untuk pembelajaran, dengan target dalam satuan 100.000 USD.


**Sebelum Run All:** nyalakan server MLflow di `http://127.0.0.1:5000` sesuai perintah pada tahap 15. Biarkan terminal server berjalan.


## 01. Pastikan environment dan dependency

**Tujuan:** memasang library pada Python yang benar-benar dipakai kernel. `sys.executable` menunjukkan lokasi Python tersebut. `find_spec` memeriksa apakah modul tersedia; instalasi dilakukan hanya jika ada modul yang belum terpasang.

Daftar versi mengikuti environment proyek yang sudah diuji. Pilih kernel Python 3.13 dari `.venv`. Jika menggunakan folder proyek, instalasi melalui `requirements-dev.txt` di terminal juga sudah mencukupi.

**Hasil yang diperiksa:** lokasi Python menunjuk environment yang diinginkan dan tidak ada error import. Bila kernel sudah memuat versi lama lalu paket berubah, restart kernel sebelum melanjutkan.


In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    "mlflow": "mlflow==3.17.2",
    "sklearn": "scikit-learn==1.9.1",
    "skops": "skops==0.16.0",
    "pandas": "pandas==3.1.0",
    "numpy": "numpy==2.5.4",
    "yaml": "PyYAML==6.0.3",
    "dvc": "dvc==3.67.1",
}
missing = [requirement for module, requirement in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
print("Python kernel:", sys.executable)


## 02. Import alat yang akan dipakai

`pandas` menangani tabel, `numpy` membantu validasi angka, dan `scikit-learn` menyediakan split, preprocessing, model, tuning, serta metrik. MLflow menyimpan hasil eksperimen dan model. `hashlib` menghitung identitas isi data, sedangkan `yaml` membaca metadata DVC.

Import dikelompokkan berdasarkan fungsi agar peserta mudah menemukan asal setiap komponen. Semua fungsi proyek akan ditulis atau dirangkai di notebook ini.


In [ ]:
import hashlib
import json
import shutil
import time
from datetime import datetime
from importlib.metadata import version
from io import BytesIO
from pathlib import Path
from uuid import uuid4

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import yaml
from mlflow.models import infer_signature
from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

for package in ["mlflow", "scikit-learn", "pandas", "numpy", "skops", "dvc"]:
    print(f"{package}: {version(package)}")


## 03. Tentukan folder kerja dan parameter umum

Jika notebook dijalankan dari folder `notebooks`, root proyek adalah folder induknya. Jika notebook dipindahkan ke folder lain, folder aktif akan menjadi root. Semua file menggunakan path absolut yang diturunkan dari root ini sehingga lokasi penyimpanannya jelas.

`RANDOM_STATE` membuat pembagian data dan model dapat diulang. `TEST_SIZE=0.3` berarti 30% data dipakai sebagai test set. `QUICK_DEMO=True` memilih grid kecil untuk kelas; ubah menjadi `False` untuk grid lengkap.

**Periksa:** path yang dicetak adalah folder proyek yang benar sebelum menjalankan cell yang menulis file.


In [ ]:
current_folder = Path.cwd().resolve()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "california_housing.csv"
DVC_PATH = DATA_PATH.with_suffix(".csv.dvc")
DATA_RELATIVE = DATA_PATH.relative_to(PROJECT_ROOT).as_posix()

RANDOM_STATE = 42
TEST_SIZE = 0.3
QUICK_DEMO = True
FEATURES = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
            "Population", "AveOccup", "Latitude", "Longitude"]
TARGET = "MedHouseVal"

print("Root proyek:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)


## 04. Tulis helper untuk menjalankan perintah

`subprocess.run` menjalankan command sebagai daftar argumen. `cwd` memastikan Git/DVC bekerja di root proyek, bukan di folder kernel yang kebetulan aktif. `check_returncode()` menghentikan cell jika command gagal, sehingga langkah berikutnya tidak memakai hasil yang belum terbentuk.

`run_dvc` selalu memakai `sys.executable -m dvc`. Ini menghindari tertukarnya DVC dari environment Python lain. Helper juga menyediakan PATH Python kernel untuk stage DVC yang menjalankan perintah `python`.


In [ ]:
import os

def run_command(*arguments):
    command_env = os.environ.copy()
    command_env["PATH"] = str(Path(sys.executable).parent) + os.pathsep + command_env["PATH"]
    completed = subprocess.run(
        list(arguments), cwd=PROJECT_ROOT, env=command_env,
        stdin=subprocess.DEVNULL, capture_output=True,
        text=True, encoding="utf-8", errors="replace",
    )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    completed.check_returncode()
    return completed.stdout.strip()


def run_dvc(*arguments):
    return run_command(sys.executable, "-m", "dvc", *arguments)


## 05. Inisialisasi Git dan DVC

Git menyimpan riwayat kode dan metadata; DVC menyimpan isi data pada cache serta menulis file rujukan `.dvc`. Setup hanya dijalankan bila belum ada. Pada folder di dalam repo Git yang lebih besar, `--subdir` menempatkan proyek DVC di folder ini.

Aturan ignore mencegah environment, cache, model lokal, dan database MLflow masuk Git. `.dvcignore` juga mengecualikan penyimpanan MLflow agar salinan metadata `.dvc` yang dicatat sebagai artifact tidak dianggap sebagai stage DVC baru.

**Hasil:** folder `.git`/`.dvc` tersedia dan versi DVC tercetak. Cell ini tidak melakukan commit atau push.


In [ ]:
if shutil.which("git") is None:
    raise RuntimeError("Git belum tersedia. Instal Git lalu buka ulang terminal/editor.")

probe = subprocess.run(["git", "rev-parse", "--show-toplevel"], cwd=PROJECT_ROOT,
                       capture_output=True, text=True)
if probe.returncode != 0:
    run_command("git", "init")
    git_root = PROJECT_ROOT
else:
    git_root = Path(probe.stdout.strip()).resolve()

if not (PROJECT_ROOT / ".dvc").exists():
    run_dvc("init", *(() if git_root == PROJECT_ROOT else ("--subdir",)))

for filename, rules in {
    ".gitignore": [".venv/", "__pycache__/", ".ipynb_checkpoints/", ".mlflow/",
                   "artifacts/", "mlruns/", "mlartifacts/", ".env", ".env.*"],
    ".dvcignore": [".venv/", "__pycache__/", ".mlflow/", "mlruns/", "mlartifacts/"],
}.items():
    path = PROJECT_ROOT / filename
    existing = path.read_text(encoding="utf-8") if path.exists() else ""
    missing_rules = [rule for rule in rules if rule not in existing.splitlines()]
    if missing_rules:
        with path.open("a", encoding="utf-8") as stream:
            stream.write(("\n" if existing and not existing.endswith("\n") else "")
                         + "\n".join(missing_rules) + "\n")
run_dvc("--version")


## 06. Siapkan CSV tanpa menimpa versi yang sudah ada

Bila CSV tersedia, notebook memakai file tersebut. Bila CSV belum ada, dataset publik California Housing diambil melalui scikit-learn. Jika metadata DVC sudah ada dari clone, hasil pengambilan data harus cocok dengan hash yang tercatat sebelum file ditulis.

Dua variasi line ending diperiksa karena snapshot awal dapat berasal dari Windows. Ini tidak mengubah angka dataset; perbedaannya berada pada byte pemisah baris. Jika hash tetap berbeda, versi tersebut harus dipulihkan dari cache/remote DVC dan cell berhenti.

**Hasil baseline:** 20.640 baris, delapan fitur, dan satu target. Unduhan pertama memerlukan internet; scikit-learn dapat menggunakan cache lokal pada pengambilan berikutnya.


In [ ]:
if not DATA_PATH.exists():
    housing = fetch_california_housing(as_frame=True)
    candidates = [housing.frame.to_csv(index=False, lineterminator=ending).encode("utf-8")
                  for ending in ["\n", "\r\n"]]
    data_bytes = candidates[0]
    if DVC_PATH.exists():
        metadata = yaml.safe_load(DVC_PATH.read_text(encoding="utf-8"))
        expected = next(item["md5"] for item in metadata["outs"]
                        if item["path"] == DATA_PATH.name)
        data_bytes = next((item for item in candidates
                           if hashlib.md5(item, usedforsecurity=False).hexdigest() == expected), None)
        if data_bytes is None:
            raise RuntimeError("Sumber publik tidak cocok dengan versi DVC. Gunakan dvc pull/checkout.")
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    DATA_PATH.write_bytes(data_bytes)
    print("CSV berhasil disiapkan.")
else:
    print("Memakai CSV yang sudah ada; isinya tidak ditimpa.")
print(DATA_PATH)


## 07. Catat versi file dengan DVC

`dvc add` menyimpan isi CSV ke cache DVC dan membuat/memperbarui file metadata. DVC juga menambahkan CSV ke `.gitignore` di folder data. Metadata berisi hash, ukuran file, dan nama data.

Jika file sudah berubah secara sengaja, menjalankan cell ini berarti mencatat versi baru sebagai versi kerja. Riwayat versinya baru tersimpan di Git setelah file `.dvc` di-commit. Jangan menyamakan `dvc add` dengan `git commit`.


In [ ]:
run_dvc("add", DATA_RELATIVE)
dataset_dvc_metadata = DVC_PATH.read_text(encoding="utf-8")
print("Metadata dataset:")
print(dataset_dvc_metadata)
run_dvc("status", DATA_RELATIVE + ".dvc")


## 08. Baca dataset dan identitas isinya

CSV dibaca menjadi byte terlebih dahulu. SHA-256 dihitung dari byte yang sama dengan yang dibaca pandas, sehingga identitas data dan DataFrame merujuk pada isi file yang sama. Hash DVC juga diperiksa untuk mendeteksi perubahan setelah cell `dvc add`.

`df.head()` hanya menampilkan beberapa baris awal untuk inspeksi. DataFrame `df` tetap berisi seluruh baris.


In [ ]:
dataset_bytes = DATA_PATH.read_bytes()
dataset_sha256 = hashlib.sha256(dataset_bytes).hexdigest()
expected_md5 = next(item["md5"] for item in yaml.safe_load(dataset_dvc_metadata)["outs"]
                    if item["path"] == DATA_PATH.name)
if hashlib.md5(dataset_bytes, usedforsecurity=False).hexdigest() != expected_md5:
    raise RuntimeError("CSV berubah setelah dvc add. Tinjau dan catat versi yang benar.")

df = pd.read_csv(BytesIO(dataset_bytes))
print("Ukuran tabel:", df.shape)
print("SHA-256:", dataset_sha256)
df.head()


## 09. Validasi sebelum training

Pemeriksaan ini memastikan fitur/target tersedia, semua kolom yang dipakai berupa angka, dan tidak ada nilai kosong atau infinity. Program berhenti lebih awal dengan pesan yang jelas bila kontrak input dilanggar.

Kita tidak menghapus baris atau mengisi missing value secara diam-diam. Perubahan pembersihan data perlu menjadi keputusan yang terlihat dan tercatat sebagai versi data baru.


In [ ]:
required_columns = FEATURES + [TARGET]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Kolom belum lengkap: {missing_columns}")
if df.empty:
    raise ValueError("Dataset kosong.")
if not all(pd.api.types.is_numeric_dtype(df[column]) for column in required_columns):
    raise ValueError("Fitur dan target harus numerik.")
if not np.isfinite(df[required_columns].to_numpy(dtype=float)).all():
    raise ValueError("Ada nilai kosong, NaN, atau infinity pada fitur/target.")
print("Validasi lolos.")
df[required_columns].describe().T


## 10. Pisahkan fitur, target, training, dan test

`X` berisi delapan fitur, sedangkan `y` berisi target yang ingin diprediksi. Target tidak ikut masuk ke fitur. `train_test_split` menyisihkan 30% data untuk evaluasi akhir dan memakai seed 42 agar pembagian dapat diulang.

Test set tidak digunakan untuk memilih hyperparameter. Pada baseline 20.640 baris, training berisi 14.448 baris dan test berisi 6.192 baris. Data pada tahap ini masih mentah, belum di-scaling.


In [ ]:
X = df.loc[:, FEATURES].copy()
y = df[TARGET].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
)
assert set(X_train.index).isdisjoint(set(X_test.index))
print("Training:", X_train.shape, "Test:", X_test.shape)
X_train.head()


## 11. Bangun pipeline preprocessing dan model

Pipeline merangkai `StandardScaler` dan `RandomForestRegressor`. Saat pipeline di-fit, scaler belajar dari data training, lalu hasil transformasinya masuk ke model. Saat `predict`, pipeline menggunakan statistik scaler yang sudah dipelajari.

Menaruh scaler di dalam pipeline juga membuat GridSearchCV melatih preprocessing hanya pada bagian training setiap fold. Ini menghindari kebocoran informasi dari validation fold. Random Forest biasanya tidak memerlukan scaling, tetapi scaler dipertahankan untuk memperjelas pola penyimpanan preprocessing bersama model.

`random_state` mengontrol pengacakan model. `n_jobs=1` pada Random Forest mencegah parallelism bertingkat ketika GridSearchCV nanti menjalankan fitting.


In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)),
])
pipeline


## 12. Tentukan kandidat hyperparameter

Awalan `model__` mengarah ke tahap bernama `model` di dalam pipeline. Mode cepat mempunyai dua kandidat dan dua fold, sehingga empat fitting cross-validation dilakukan. Grid penuh mempunyai 24 kandidat dan tiga fold, sehingga 72 fitting dilakukan, ditambah refit model pemenang.

- `n_estimators`: jumlah pohon.
- `max_depth`: batas kedalaman pohon; `None` tidak memberi batas ini.
- `min_samples_split`: minimal sampel agar node dapat dibagi.
- `min_samples_leaf`: minimal sampel pada leaf.

`neg_mean_squared_error` dipakai karena GridSearchCV selalu memilih skor yang lebih besar. Tanda negatif dibalik saat menampilkan CV MSE.


In [ ]:
if QUICK_DEMO:
    param_grid = {"model__n_estimators": [20], "model__max_depth": [5, 10],
                  "model__min_samples_split": [2], "model__min_samples_leaf": [1]}
    CV_FOLDS = 2
else:
    param_grid = {"model__n_estimators": [100, 200], "model__max_depth": [5, 10, None],
                  "model__min_samples_split": [2, 5], "model__min_samples_leaf": [1, 2]}
    CV_FOLDS = 3

candidate_count = int(np.prod([len(values) for values in param_grid.values()]))
print("Kandidat:", candidate_count, "Fold:", CV_FOLDS,
      "Total fitting CV:", candidate_count * CV_FOLDS)


## 13. Jalankan GridSearchCV

`fit` menerima training set saja. GridSearchCV membagi training set menjadi fold, mencoba semua kandidat, menghitung skor rata-rata, lalu melatih kembali pipeline dengan parameter pemenang pada seluruh training set.

Hasil terbaik di sini berarti terbaik di antara kandidat dan skema cross-validation yang digunakan. Ini belum merupakan keputusan otomatis bahwa model siap untuk semua penggunaan bisnis.

Durasi fitting dicatat terpisah. Cell logging MLflow diletakkan setelah training agar error penyimpanan tidak memaksa peserta mengulang seluruh tuning.


In [ ]:
grid_search = GridSearchCV(
    estimator=pipeline, param_grid=param_grid, cv=CV_FOLDS,
    scoring="neg_mean_squared_error", n_jobs=1, verbose=1, error_score="raise",
)
started = time.perf_counter()
grid_search.fit(X_train, y_train)
training_seconds = time.perf_counter() - started
best_model = grid_search.best_estimator_
best_params = {key.removeprefix("model__"): value
               for key, value in grid_search.best_params_.items()}
print("Parameter terbaik:", best_params)
print("CV MSE:", -grid_search.best_score_)
print(f"Durasi training: {training_seconds:.2f} detik")


## 14. Evaluasi model dan baseline sederhana

Prediksi dibuat pada test set mentah; pipeline melakukan scaling secara otomatis. MSE menghitung rata-rata kesalahan kuadrat, RMSE mengembalikan skala ke satuan target, dan R² membandingkan performa dengan prediksi rata-rata. Nilai R² dapat negatif.

`DummyRegressor(strategy="mean")` memprediksi rata-rata target training untuk setiap sampel. Baseline ini membantu peserta menilai apakah model kompleks memberikan manfaat dibanding tebakan sederhana. Model dan baseline dinilai pada test set yang sama.


In [ ]:
y_pred = best_model.predict(X_test)
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
mse = float(mean_squared_error(y_test, y_pred))
metrics = {
    "mse": mse,
    "rmse": float(np.sqrt(mse)),
    "r2_score": float(r2_score(y_test, y_pred)),
    "cv_mse": float(-grid_search.best_score_),
    "baseline_mse": float(mean_squared_error(y_test, baseline_pred)),
    "training_seconds": training_seconds,
}
pd.Series(metrics, name="nilai")


## 15. Hubungkan ke server MLflow di port 5000

Notebook memakai `http://127.0.0.1:5000`. Sebelum menjalankan notebook, buka terminal PowerShell kedua **di root proyek** dan jalankan:

```powershell
New-Item -ItemType Directory -Force .mlflow/server
.\.venv\Scripts\python.exe -m mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///.mlflow/server/mlflow.db --artifacts-destination .mlflow/server/artifacts --serve-artifacts
```

Biarkan terminal server terbuka. Buka `http://127.0.0.1:5000` untuk melihat UI. Server menerima catatan run dan file model; SQLite di belakang server menyimpan metadata, sedangkan folder artifacts menyimpan file. Notebook mengakses server melalui HTTP.

Cell berikut memeriksa koneksi dengan batas waktu singkat. Jika server belum aktif, jalankan perintah di atas lalu ulangi cell ini. Pengaturan tracking dilakukan sebelum `start_run`. Environment variable `MLFLOW_TRACKING_URI` hanya diperlukan untuk mengganti alamat secara sengaja, misalnya pada pengujian otomatis.

In [ ]:
from urllib.error import URLError
from urllib.request import urlopen

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT_NAME = "house-price-all-in-one"
if TRACKING_URI.startswith(("http://", "https://")):
    try:
        with urlopen(TRACKING_URI.rstrip("/") + "/health", timeout=5) as response:
            if response.status != 200:
                raise RuntimeError("Server MLflow belum siap.")
    except (URLError, TimeoutError) as error:
        raise RuntimeError(
            f"MLflow belum dapat diakses di {TRACKING_URI}. "
            "Jalankan server sesuai petunjuk Markdown, lalu ulangi cell ini."
        ) from error

mlflow.set_tracking_uri(TRACKING_URI)
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    artifact_location = None
    if TRACKING_URI.startswith("sqlite:///"):
        db_path = Path(TRACKING_URI.removeprefix("sqlite:///"))
        artifact_location = (db_path.parent / "artifacts").resolve().as_uri()
    mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=artifact_location)
mlflow.set_experiment(EXPERIMENT_NAME)
print("Tracking URI:", TRACKING_URI)
print("Experiment:", EXPERIMENT_NAME)

## 16. Log parameter, metrik, versi data, dan model

Sebelum logging, checksum file dan metadata diperiksa kembali agar tidak mengaitkan model dengan versi data yang sudah berubah. Parameter menjelaskan konfigurasi; metrik menjelaskan hasil; tag dan artifact menghubungkan run ke dataset.

`infer_signature` mencatat bentuk input/output model. `input_example` memberikan contoh fitur mentah. Model disimpan memakai `skops`. Tipe `sklearn.tree._tree.Tree` dipercaya secara eksplisit karena pipeline ini dilatih sendiri di atas; kita tidak memasukkan semua tipe tidak dikenal secara otomatis.

`model_info.model_uri` menunjuk model yang benar-benar tersimpan pada run ini. Simpan URI tersebut, bukan menebak path artifact.


In [ ]:
if hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() != dataset_sha256:
    raise RuntimeError("Dataset berubah setelah training. Ulangi proses dengan versi yang benar.")
if DVC_PATH.read_text(encoding="utf-8") != dataset_dvc_metadata:
    raise RuntimeError("Metadata DVC berubah setelah training.")

signature = infer_signature(X_train, best_model.predict(X_train))
model_options = {
    "signature": signature,
    "input_example": X_train.head(3),
    "serialization_format": "skops",
    "skops_trusted_types": ["sklearn.tree._tree.Tree"],
    "pip_requirements": [f"{name}=={version(name)}"
                         for name in ["mlflow", "scikit-learn", "skops", "pandas", "numpy"]],
}
with mlflow.start_run() as run:
    mlflow.log_params({**best_params, "random_state": RANDOM_STATE,
                       "test_size": TEST_SIZE, "cv": CV_FOLDS})
    mlflow.log_metrics(metrics)
    mlflow.set_tags({"dataset.sha256": dataset_sha256, "dataset.versioning": "dvc"})
    mlflow.log_text(dataset_dvc_metadata, "data_version/california_housing.csv.dvc")
    model_info = mlflow.sklearn.log_model(
        best_model, name="model", registered_model_name="Best Random Forest Model",
        **model_options,
    )
    RUN_ID = run.info.run_id
print("Run ID:", RUN_ID)
print("Model URI:", model_info.model_uri)


## 17. Simpan model portabel dan laporan run

MLflow sudah menyimpan model pada tracking store. Salinan lokal berikut memudahkan pemakaian tanpa harus mengakses tracking server. Nama folder mengandung waktu dan kode acak pendek sehingga menjalankan ulang cell tidak menimpa hasil sebelumnya.

`run.json` mencatat lokasi model, tracking URI, checksum dataset, parameter, dan metrik. Setelah kernel restart, laporan ini menjadi titik masuk untuk menemukan model kembali. Pada konfigurasi server, gunakan URL tanpa kredensial di dalam URI; autentikasi sebaiknya melalui environment yang sesuai.


In [ ]:
run_label = datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "all_in_one" / run_label
OUTPUT_DIR.mkdir(parents=True)
mlflow.sklearn.save_model(best_model, path=str(OUTPUT_DIR / "model"), **model_options)
run_report = {
    "run_id": RUN_ID, "model_uri": model_info.model_uri, "tracking_uri": TRACKING_URI,
    "local_model_path": str(OUTPUT_DIR / "model"), "dataset_sha256": dataset_sha256,
    "best_params": best_params, "metrics": metrics,
    "random_state": RANDOM_STATE, "test_size": TEST_SIZE,
}
RUN_REPORT_PATH = OUTPUT_DIR / "run.json"
RUN_REPORT_PATH.write_text(json.dumps(run_report, indent=2), encoding="utf-8")
print("Simpan path ini untuk load setelah restart kernel:")
print(RUN_REPORT_PATH)


## 18. Load kembali dari MLflow dan dari folder model

Cell ini membaca laporan run dari disk, mengatur tracking URI yang benar, lalu memuat model. Setelah restart kernel, import `json`, `Path`, dan `mlflow.sklearn`, kemudian isi `RUN_REPORT_PATH` dengan path laporan yang sudah dicetak. Cell logging/training tidak perlu dijalankan lagi hanya untuk inference.

Dua objek dimuat: satu melalui URI MLflow, satu melalui folder model portabel. Keduanya mencakup scaler dan Random Forest.


In [ ]:
saved_run = json.loads(Path(RUN_REPORT_PATH).read_text(encoding="utf-8"))
mlflow.set_tracking_uri(saved_run["tracking_uri"])
loaded_best_model = mlflow.sklearn.load_model(saved_run["model_uri"])
loaded_local_model = mlflow.sklearn.load_model(saved_run["local_model_path"])
print("Model dari MLflow:", saved_run["model_uri"])
print("Model lokal:", saved_run["local_model_path"])


## 19. Buktikan hasil prediksi tetap sama

Pengujian ini membandingkan prediksi sebelum penyimpanan, sesudah load dari MLflow, dan sesudah load dari folder lokal pada input yang sama. `assert_allclose` akan menghentikan cell bila ada perbedaan di luar toleransi numerik.

Tabel aktual versus prediksi membantu inspeksi. Pemeriksaan kesamaan hasil membuktikan proses simpan/load bekerja; itu berbeda dari pembuktian kualitas model untuk kasus penggunaan baru.


In [ ]:
y_pred_loaded = loaded_best_model.predict(X_test)
y_pred_local = loaded_local_model.predict(X_test)
np.testing.assert_allclose(y_pred, y_pred_loaded, rtol=1e-12, atol=1e-12)
np.testing.assert_allclose(y_pred, y_pred_local, rtol=1e-12, atol=1e-12)
print("PASS: prediksi sebelum dan sesudah load sama.")
print("MSE setelah load:", mean_squared_error(y_test, y_pred_loaded))
pd.DataFrame({"Aktual": y_test, "Prediksi": y_pred_loaded}).head(10)


## 20. Contoh inference dan ekspor hasil

Contoh memakai lima baris test set sebagai simulasi input baru. Fitur tetap mentah dan disusun sesuai urutan yang disimpan model. Target tidak diberikan kepada model.

Pipeline mengerjakan preprocessing secara otomatis. Hasil disimpan di folder run sebagai CSV. Angka prediksi masih dalam satuan target dataset; kolom USD di bawah hanya mengalikan satuannya dengan 100.000.


In [ ]:
sample_input = X_test.head(5).loc[:, FEATURES].copy()
if not np.isfinite(sample_input.to_numpy(dtype=float)).all():
    raise ValueError("Input prediksi tidak valid.")
sample_predictions = loaded_best_model.predict(sample_input)
prediction_table = pd.DataFrame({
    "prediction": sample_predictions,
    "prediction_usd_dataset_unit": sample_predictions * 100_000,
}, index=sample_input.index)
prediction_table.to_csv(OUTPUT_DIR / "sample_predictions.csv", index=False)
print("Hasil:", OUTPUT_DIR / "sample_predictions.csv")
prediction_table


## 21. Periksa run di MLflow UI

Buka `http://127.0.0.1:5000` di browser. Server yang dijalankan sebelum training menyediakan UI dan API tracking pada alamat yang sama.

Pilih experiment `house-price-all-in-one`, lalu periksa parameter, metrik, tag `dataset.sha256`, metadata `.dvc`, dan model. Cocokkan run ID dengan `RUN_ID` di notebook. `loaded_best_model` dimuat dari model URI pada server yang sama.

Terminal server dibiarkan berjalan selama logging atau load dari MLflow. Hentikan dengan Ctrl+C setelah selesai. Folder `.mlflow/server` menyimpan metadata dan artifacts agar run tetap tersedia saat server dinyalakan lagi dengan perintah yang sama. Run lama dari database SQLite lain tidak otomatis digabungkan ke server ini.

## 22. Simpan dan pulihkan riwayat data

Perintah berikut dijalankan secara sengaja di terminal. `dvc add` pada cell sebelumnya sudah menyimpan isi CSV ke cache; Git menyimpan rujukan versinya.

```powershell
git add .dvc .dvcignore .gitignore data/raw/.gitignore data/raw/california_housing.csv.dvc notebooks/experiments_all_in_one.ipynb
git commit -m "Record all-in-one notebook and dataset version"
git tag data-all-in-one-v1
```

Tag dibuat sekali. Setelah ada perubahan data yang disengaja, jalankan kembali cell DVC dan pembacaan data, lalu commit metadata versi baru. Untuk memulihkan baseline setelah perubahan penting sudah disimpan:

```powershell
git restore --source=data-all-in-one-v1 -- data/raw/california_housing.csv.dvc
.\.venv\Scripts\python.exe -m dvc checkout data/raw/california_housing.csv.dvc
```

Setelah checkout, baca ulang data dan ulangi split/training jika ingin membuat model untuk versi tersebut. Variabel yang sudah berada di memori tidak berubah otomatis. Untuk berbagi versi modifikasi ke komputer lain, siapkan remote DVC lalu `dvc push`; file Git saja tidak berisi CSV.


## 23. Hubungkan ke struktur kode dan CI/CD

Notebook ini tetap lengkap tanpa modul proyek. Versi modular memindahkan tanggung jawab yang sama ke `data.py`, `config.py`, `train.py`, `predict.py`, dan `cli.py`. Ini memudahkan tes, CLI, serta container, sementara notebook all-in-one membantu memahami proses dari awal.

Workflow GitHub Actions mempunyai tiga tahap: `quality` memeriksa Python/tes, `container` membangun dan menguji image, dan `publish` mengirim image yang lolos ke GHCR. Push ke branch utama/tag rilis dapat memicu publikasi; pull request menjalankan pemeriksaan. Docker image berisi aplikasi modular, sedangkan model notebook tetap dapat digunakan sebagai folder model MLflow yang portabel.

Panduan detail setiap blok kode, perintah Docker, GitHub Actions, latihan DVC, dan troubleshooting tersedia pada panduan step-by-step terpisah. Gunakan notebook modular dan all-in-one sebagai dua pendekatan belajar, bukan menjalankan keduanya bersamaan dengan asumsi variabelnya saling berbagi.
